In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime, gc
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import Word2Vec
from pyspark.sql.types import StringType, ArrayType, DoubleType

# Cấu hình đường dẫn
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
ARTICLES_FILE = BASE_PATH + "processed/articles_processed.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Khởi tạo Spark tối ưu RAM
spark = SparkSession.builder \
    .appName("HM_Metadata_Pro_Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# 1. Xác định mốc thời gian động (6-1-1)
transactions = spark.read.parquet(INPUT_FILE)
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
test_start_date = max_date - datetime.timedelta(days=7)
val_start_date = test_start_date - datetime.timedelta(days=7)

# Lọc dữ liệu 6 tuần đầu (W1-W6) để làm Train/Profile
train_data = transactions.filter(F.col("t_dat") < F.lit(val_start_date)) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

articles = spark.read.parquet(ARTICLES_FILE) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

print(f"✅ Spark Ready! Đánh giá trên tuần: {val_start_date} -> {test_start_date}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Spark Ready! Đánh giá trên tuần: 2020-09-07 17:00:00 -> 2020-09-14 17:00:00


In [ ]:
# 1. Tìm Category yêu thích nhất của mỗi khách (Aggregation)
user_fav_cat = train_data.join(F.broadcast(articles.select("article_id", "product_group_name")), "article_id") \
    .groupBy("customer_id", "product_group_name").count()

window_fav = Window.partitionBy("customer_id").orderBy(F.desc("count"))
user_profile = user_fav_cat.withColumn("rn", F.row_number().over(window_fav)) \
    .filter(F.col("rn") == 1).select("customer_id", "product_group_name")

# 2. Tìm Top 10 món bán chạy nhất của mỗi Category trong tuần gần nhất (Seasonal)
w6_start = val_start_date - datetime.timedelta(days=7)
trending_items = train_data.filter(F.col("t_dat") >= F.lit(w6_start)) \
    .join(F.broadcast(articles.select("article_id", "product_group_name")), "article_id") \
    .groupBy("product_group_name", "article_id").count()

window_pop = Window.partitionBy("product_group_name").orderBy(F.desc("count"))
top_trending = trending_items.withColumn("pop_rank", F.row_number().over(window_pop)) \
    .filter(F.col("pop_rank") <= 10) \
    .select("product_group_name", F.col("article_id").alias("article_id"))

# 3. Kết hợp: Gợi ý món Hot theo đúng Gu của khách
cand_agg_seasonal = user_profile.join(F.broadcast(top_trending), "product_group_name") \
    .select("customer_id", "article_id", F.lit(1).alias("priority")) # Ưu tiên 1

print(f"✅ Đã tạo xong ứng viên Aggregation & Seasonal.")

✅ Đã tạo xong ứng viên Aggregation & Seasonal.


In [ ]:
# 1. Huấn luyện Word2Vec trên chuỗi hành vi của khách
user_sequences = train_data.join(F.broadcast(articles), "article_id") \
    .groupBy("customer_id").agg(F.collect_list("product_type_name").alias("type_list"))

w2v = Word2Vec(vectorSize=16, minCount=1, inputCol="type_list", outputCol="vector")
w2v_model = w2v.fit(user_sequences)

# 2. Lấy món cuối cùng khách mua và tìm 10 món tương đồng nhất về ngữ nghĩa
window_last = Window.partitionBy("customer_id").orderBy(F.desc("t_dat"))
last_items = train_data.withColumn("rn", F.row_number().over(window_last)) \
    .filter(F.col("rn") == 1).select("customer_id", "article_id")

# (Để tiết kiệm RAM, ta dùng Metadata Similarity đã tính ở turn trước cho phần này)
meta_lookup = spark.read.parquet(BASE_PATH + "processed/meta_item_lookup_30.parquet")
cand_w2v = last_items.join(F.broadcast(meta_lookup), last_items.article_id == meta_lookup.seed_article_id) \
    .select("customer_id", F.explode("meta_candidates").alias("article_id"), F.lit(2).alias("priority"))

print("✅ Đã tích hợp logic Ngữ nghĩa sản phẩm.")

✅ Đã tích hợp logic Ngữ nghĩa sản phẩm.


In [ ]:
# 1. Ưu tiên 0: Cùng mẫu khác màu (Mạnh nhất)
last_items_7 = last_items.withColumn("p_code", F.substring(F.col("article_id"), 1, 7))
all_variants = articles.select(F.col("article_id").alias("v_id"), F.substring(F.col("article_id"), 1, 7).alias("p_code"))

cand_same_model = last_items_7.join(F.broadcast(all_variants), "p_code") \
    .filter(F.col("article_id") != F.col("v_id")) \
    .select("customer_id", F.col("v_id").alias("article_id"), F.lit(0).alias("priority"))

# 2. Gộp tất cả các nguồn (0: Same Model, 1: Agg/Pop, 2: Metadata/W2V)
all_candidates = cand_same_model.union(cand_agg_seasonal).union(cand_w2v)

# 3. Lấy Top 30 duy nhất cho mỗi khách
final_window = Window.partitionBy("customer_id").orderBy("priority")
meta_candidates_pro = all_candidates.dropDuplicates(["customer_id", "article_id"]) \
    .withColumn("rank", F.row_number().over(final_window)) \
    .filter(F.col("rank") <= 30) \
    .groupBy("customer_id").agg(F.collect_list("article_id").alias("meta_candidates"))

# 4. Lưu file cuối cùng
meta_candidates_pro.write.mode("overwrite").parquet(OUTPUT_DIR + "meta_candidates_pro_W7.parquet")
print(f"🏆 Xuất file thành công: meta_candidates_pro_W7.parquet")

🏆 Xuất file thành công: meta_candidates_pro_W7.parquet


In [ ]:
# 1. Thực tế mua ở Tuần 7
ground_truth = transactions.filter((F.col("t_dat") >= F.lit(val_start_date)) & (F.col("t_dat") < F.lit(test_start_date))) \
    .select("customer_id", F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id"))

actual_counts = ground_truth.groupBy("customer_id").count().withColumnRenamed("count", "actual_cnt")

# 2. Join với ứng viên Pro
candidates_exploded = meta_candidates_pro.select("customer_id", F.explode("meta_candidates").alias("article_id"))
hits = ground_truth.join(candidates_exploded, ["customer_id", "article_id"], "inner") \
    .groupBy("customer_id").count().withColumnRenamed("count", "hit_cnt")

# 3. Tính Recall trung bình
recall_df = actual_counts.join(hits, "customer_id", "left").fillna(0)
avg_recall = recall_df.select(F.avg(F.col("hit_cnt") / F.col("actual_cnt"))).collect()[0][0]

print("-" * 50)
print(f"📊 KẾT QUẢ RECALL METADATA PRO (Word2Vec + Agg + Seasonal)")
print(f"Average Recall@30: {avg_recall:.6f}")
print("-" * 50)

--------------------------------------------------
📊 KẾT QUẢ RECALL METADATA PRO (Word2Vec + Agg + Seasonal)
Average Recall@30: 0.027123
--------------------------------------------------


In [ ]:
# 1. CHUẨN BỊ GROUND TRUTH (TUẦN 7)
ground_truth = transactions.filter((F.col("t_dat") >= F.lit(val_start_date)) & (F.col("t_dat") < F.lit(test_start_date))) \
    .select("customer_id", F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id"))

actual_counts = ground_truth.groupBy("customer_id").count().withColumnRenamed("count", "actual_cnt")

# 2. HÀM TÍNH RECALL CHO TỪNG NGUỒN
def evaluate_source(source_df, source_name):
    # Lấy ứng viên của nguồn đó
    cand_exploded = source_df.select("customer_id", "article_id")

    # Tính Hits
    hits = ground_truth.join(cand_exploded, ["customer_id", "article_id"], "inner") \
        .groupBy("customer_id").count().withColumnRenamed("count", "hit_cnt")

    # Tính Recall trung bình
    recall_df = actual_counts.join(hits, "customer_id", "left").fillna(0)
    res = recall_df.select(F.avg(F.col("hit_cnt") / F.col("actual_cnt"))).collect()[0][0]
    return res

# 3. CHẠY ĐÁNH GIÁ TỪNG PHẦN
print("🚀 Đang phân tích hiệu quả từng kỹ thuật...")

# Nguồn 0: Same Model (7-digit) - Chỉ lấy tối đa 10 món đầu tiên của nguồn này
window_0 = Window.partitionBy("customer_id").orderBy(F.lit(1))
source_0 = cand_same_model.withColumn("rn", F.row_number().over(window_0)).filter(F.col("rn") <= 10)
recall_0 = evaluate_source(source_0, "Same Model")

# Nguồn 1: Aggregation & Seasonal - Chỉ lấy tối đa 10 món đầu tiên
source_1 = cand_agg_seasonal.withColumn("rn", F.row_number().over(window_0)).filter(F.col("rn") <= 10)
recall_1 = evaluate_source(source_1, "Agg & Seasonal")

# Nguồn 2: Word2Vec/Metadata Similarity - Chỉ lấy tối đa 10 món đầu tiên
source_2 = cand_w2v.withColumn("rn", F.row_number().over(window_0)).filter(F.col("rn") <= 10)
recall_2 = evaluate_source(source_2, "Word2Vec/Meta")

# 4. IN BẢNG SO SÁNH
print("\n" + "="*50)
print(f"{'Kỹ thuật Metadata':<30} | {'Recall@10':<10}")
print("-" * 50)
print(f"{'1. Same Model (7-digit)':<30} | {recall_0:.6f}")
print(f"{'2. Aggregation & Seasonal':<30} | {recall_1:.6f}")
print(f"{'3. Word2Vec / Similarity':<30} | {recall_2:.6f}")
print("="*50)

🚀 Đang phân tích hiệu quả từng kỹ thuật...

Kỹ thuật Metadata              | Recall@10 
--------------------------------------------------
1. Same Model (7-digit)        | 0.009449
2. Aggregation & Seasonal      | 0.016333
3. Word2Vec / Similarity       | 0.000461
